In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

In [3]:
# 1. Simple Neural Network to approximate Q-values
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.fc(x)

In [4]:
from collections import deque
# 2. Replay Buffer (Standard)
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    def push(self, *args): 
        self.buffer.append(args)
    def sample(self, batch_size): 
        return random.sample(self.buffer, batch_size)
    def __len__(self):
        return len(self.buffer)

In [ ]:

import gymnasium as gym
import highway_env


def train(episodes=500):
    env = gym.make('highway-v0')
    state_dim = np.prod(env.observation_space.shape)
    action_dim = env.action_space.n
    
    q_net = QNetwork(state_dim, action_dim)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    q_net = q_net.to(device)

    print(f"Training on: {device}")
    print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

    optimizer = optim.Adam(q_net.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    # Hyperparameters
    gamma = 0.99
    epsilon = 1.0
    epsilon_decay = 0.995
    episodes = episodes

    for ep in range(episodes):
        state, _ = env.reset()
        done = False

        total_loss = 0
        total_reward = 0
        
        while not done:
            state_tensor = torch.FloatTensor(state.flatten()).to(device)
            
            # Epsilon-greedy action selection
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = q_net(state_tensor).argmax().item()

            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated

            # Prepare Target 
            next_state_tensor = torch.FloatTensor(next_state.flatten()).to(device)
            
            with torch.no_grad():
                next_q_values = q_net(next_state_tensor)
                max_next_q = torch.max(next_q_values)
                # Bellman Target: r + gamma * max(Q(s', a'))
                target_q = reward + (gamma * max_next_q * (1 - terminated))
                target_q = target_q.to(device)

            # Current Q-value prediction
            current_q = q_net(state_tensor)[action]

            # Backpropagation
            loss = loss_fn(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            total_loss += loss.item()
            optimizer.step()

            state = next_state

        epsilon = max(0.01, epsilon * epsilon_decay)
        
        
        print(f"Episode {ep+1} | Reward: {total_reward:.2f} | Loss: {total_loss:.4f} | Epsilon: {epsilon:.2f}")

    env.close()
    return q_net


In [13]:
train()

Training on: cuda
GPU name: NVIDIA GeForce RTX 5060 Laptop GPU
Episode 1 | Reward: 8.10 | Loss: 8.6968 | Epsilon: 0.99
Episode 2 | Reward: 39.44 | Loss: 43.7105 | Epsilon: 0.99
Episode 3 | Reward: 41.35 | Loss: 46.3823 | Epsilon: 0.99
Episode 4 | Reward: 45.19 | Loss: 50.9184 | Epsilon: 0.98
Episode 5 | Reward: 50.63 | Loss: 57.1219 | Epsilon: 0.98
Episode 6 | Reward: 71.86 | Loss: 83.2511 | Epsilon: 0.97
Episode 7 | Reward: 79.42 | Loss: 92.2359 | Epsilon: 0.97
Episode 8 | Reward: 93.58 | Loss: 116.0862 | Epsilon: 0.96
Episode 9 | Reward: 123.70 | Loss: 172.5802 | Epsilon: 0.96
Episode 10 | Reward: 125.63 | Loss: 186.0797 | Epsilon: 0.95
Episode 11 | Reward: 130.67 | Loss: 209.7113 | Epsilon: 0.95
Episode 12 | Reward: 133.59 | Loss: 233.3414 | Epsilon: 0.94
Episode 13 | Reward: 144.56 | Loss: 266.0371 | Epsilon: 0.94
Episode 14 | Reward: 165.77 | Loss: 332.3183 | Epsilon: 0.93
Episode 15 | Reward: 171.98 | Loss: 372.5415 | Epsilon: 0.93
Episode 16 | Reward: 193.01 | Loss: 481.6904 | E

KeyboardInterrupt: 

## 1st Improvement: Target Q-Network

In [ ]:

import gymnasium as gym
import highway_env


def train_target_qnetwork(episodes=500):
    env = gym.make('highway-v0')
    state_dim = np.prod(env.observation_space.shape)
    action_dim = env.action_space.n

    policy_net = QNetwork(state_dim, action_dim)
    target_net = QNetwork(state_dim, action_dim)
    target_net.load_state_dict(policy_net.state_dict())

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    policy_net = policy_net.to(device)
    target_net = target_net.to(device)

    print(f"Training on: {device}")
    print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

    optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    # Hyperparameters
    gamma = 0.99
    epsilon = 1.0
    epsilon_decay = 0.995
    episodes = episodes

    target_update_freq = 10

    for ep in range(episodes):
        total_loss = 0
        total_reward = 0
        state, _ = env.reset()
        done = False
        step_count = 0
        
        while not done:
            step_count += 1
            state_tensor = torch.FloatTensor(state.flatten()).to(device)
            
            # Epsilon-greedy action selection
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = policy_net(state_tensor).argmax().item()

            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated

            # Prepare Target 
            next_state_tensor = torch.FloatTensor(next_state.flatten()).to(device)
            
            with torch.no_grad():
                next_q_values = target_net(next_state_tensor)
                max_next_q = torch.max(next_q_values)
                # Bellman Target: r + gamma * max(Q(s', a'))
                target_q = reward + (gamma * max_next_q * (1 - terminated))
                target_q = target_q.to(device)

            # Current Q-value prediction
            current_q = policy_net(state_tensor)[action]

            # Backpropagation
            loss = loss_fn(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            total_loss += loss.item()
            optimizer.step()

            state = next_state

            avg_reward = total_reward / step_count
            avg_loss = total_loss / step_count

        if ep % target_update_freq == 0:
            target_net.load_state_dict(policy_net.state_dict())

        epsilon = max(0.01, epsilon * epsilon_decay)
        
        
        print(f"Episode {ep+1} | Average Reward: {avg_reward:.2f} | Average Loss: {avg_loss:.4f} | Epsilon: {epsilon:.2f}")

    env.close()
    return policy_net
